# Baseline Analysis for a Popularity Based Recommendation System 

In [1]:
from pathlib import Path
import sys

project_root = Path.cwd().parent
sys.path.append(str(project_root))

import pandas as pd

from src.data import load_movies
from src.recommenders import recommend_popular_movies

train = pd.read_csv(project_root / "data/processed/train_ratings.csv")
test = pd.read_csv(project_root / "data/processed/test_ratings.csv")
movies = load_movies()

# Look at what some of these recommendations look like

In [2]:
sample_users = [1, 10, 100]

for user_id in sample_users:
    print(f"Recommendations for user {user_id}")
    display(recommend_popular_movies(train, movies, user_id=user_id, k=10))

Recommendations for user 1


,movieId,title,genres,positive_ratings
0,318,"Shawshank Redemption, The (1994)",Crime|Drama,240
1,593,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller,205
2,527,Schindler's List (1993),Drama|War,146
3,858,"Godfather, The (1972)",Crime|Drama,144
4,4993,"Lord of the Rings: The Fellowship of the Ring,...",Adventure|Fantasy,138
5,589,Terminator 2: Judgment Day (1991),Action|Sci-Fi,134
6,7153,"Lord of the Rings: The Return of the King, The...",Action|Adventure|Drama|Fantasy,133
7,5952,"Lord of the Rings: The Two Towers, The (2002)",Adventure|Fantasy,121
8,47,Seven (a.k.a. Se7en) (1995),Mystery|Thriller,118
9,150,Apollo 13 (1995),Adventure|Drama|IMAX,117


Recommendations for user 10


,movieId,title,genres,positive_ratings
0,318,"Shawshank Redemption, The (1994)",Crime|Drama,240
1,593,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller,205
2,260,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,190
3,110,Braveheart (1995),Action|Drama|War,158
4,1196,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Sci-Fi,155
5,50,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,146
6,527,Schindler's List (1993),Drama|War,146
7,858,"Godfather, The (1972)",Crime|Drama,144
8,1198,Raiders of the Lost Ark (Indiana Jones and the...,Action|Adventure,143
9,1210,Star Wars: Episode VI - Return of the Jedi (1983),Action|Adventure|Sci-Fi,140


Recommendations for user 100


,movieId,title,genres,positive_ratings
0,318,"Shawshank Redemption, The (1994)",Crime|Drama,240
1,2571,"Matrix, The (1999)",Action|Sci-Fi|Thriller,209
2,593,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller,205
3,260,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,190
4,2959,Fight Club (1999),Action|Crime|Drama|Thriller,161
5,110,Braveheart (1995),Action|Drama|War,158
6,1196,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Sci-Fi,155
7,50,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,146
8,527,Schindler's List (1993),Drama|War,146
9,858,"Godfather, The (1972)",Crime|Drama,144


The popularity baseline recommends broadly like movies and is easy to explain, but most of the users get the same recommendations, except for the movies they have already rated. Therefore, this baseline lacks a personalized element. 

# Precision@K Evaluation
* Precious at 10 essentially evaluates, what fraction of the 10 recommendations for a user were later rated positively. 

In [3]:
from src.metrics import precision_at_k

user_id = 1
k = 10

recs = recommend_popular_movies(train, movies, user_id=user_id, k=k)

recommended_items = recs["movieId"].tolist()

relevant_items = set(
    test[
        (test["userId"] == user_id) &
        (test["is_positive"])
    ]["movieId"]
)

precision = precision_at_k(
    recommended_items=recommended_items,
    relevant_items=relevant_items,
    k=k,
)

precision

0.3

For userId = 1, of the 10 popular recommendations based on popularity, 3 were later ranked positively. 

In [4]:
k = 10
user_scores = []

test_users = test["userId"].unique()

for user_id in test_users:
    recs = recommend_popular_movies(
        train_ratings=train,
        movies=movies,
        user_id=user_id,
        k=k,
    )

    recommended_items = recs["movieId"].tolist()

    relevant_items = set(
        test[
            (test["userId"] == user_id) &
            (test["is_positive"])
        ]["movieId"]
    )

    score = precision_at_k(
        recommended_items=recommended_items,
        relevant_items=relevant_items,
        k=k,
    )

    user_scores.append(
        {
            "userId": user_id,
            "precision_at_10": score,
            "n_future_likes": len(relevant_items),
        }
    )

precision_results = pd.DataFrame(user_scores)

precision_results.head()

,userId,precision_at_10,n_future_likes
0,1,0.3,38
1,2,0.0,4
2,3,0.0,0
3,4,0.0,21
4,5,0.0,5


In [5]:
precision_results["precision_at_10"].describe()

count    610.000000
mean       0.057869
std        0.111323
min        0.000000
25%        0.000000
50%        0.000000
75%        0.100000
max        0.800000
Name: precision_at_10, dtype: float64

The popularity baseline is easy to explain but weakly personalized. Average Precision@10 is 0.058, meaning fewer than one of the top ten recommendations matches a future positive rating for the average user. The median user receives zero future-positive hits, suggesting that popularity alone is not sufficient for many users.

# Recall@k Evaluation
* Out of the number of future rated positively movies from a user, what fraction of them were recommended. 

In [6]:
from src.metrics import recall_at_k

k = 10
recall_scores = []

test_users = test["userId"].unique()

for user_id in test_users:
    recs = recommend_popular_movies(
        train_ratings=train,
        movies=movies,
        user_id=user_id,
        k=k,
    )

    recommended_items = recs["movieId"].tolist()

    relevant_items = set(
        test[
            (test["userId"] == user_id) &
            (test["is_positive"])
        ]["movieId"]
    )

    recall = recall_at_k(
        recommended_items=recommended_items,
        relevant_items=relevant_items,
        k=k,
    )

    recall_scores.append(
        {
            "userId": user_id,
            "recall_at_10": recall,
            "n_future_likes": len(relevant_items),
        }
    )

recall_results = pd.DataFrame(recall_scores)

recall_results["recall_at_10"].describe()


count    610.000000
mean       0.049672
std        0.110434
min        0.000000
25%        0.000000
50%        0.000000
75%        0.041454
max        1.000000
Name: recall_at_10, dtype: float64

On average, the popularity recommender recovered about 5.0% of each user's future-liked movies. Hopefully, this can also be improved upon.

# NDCG@K Evaluation
* Did we put relevant movies near the top of the recommendation list? Gives more credit for hits near rank 1 than hits near rank 10 

In [7]:
from src.metrics import ndcg_at_k

k = 10
ndcg_scores = []

test_users = test["userId"].unique()

for user_id in test_users:
    recs = recommend_popular_movies(
        train_ratings=train,
        movies=movies,
        user_id=user_id,
        k=k,
    )

    recommended_items = recs["movieId"].tolist()

    relevant_items = set(
        test[
            (test["userId"] == user_id) &
            (test["is_positive"])
        ]["movieId"]
    )

    ndcg = ndcg_at_k(
        recommended_items=recommended_items,
        relevant_items=relevant_items,
        k=k,
    )

    ndcg_scores.append(
        {
            "userId": user_id,
            "ndcg_at_10": ndcg,
            "n_future_likes": len(relevant_items),
        }
    )

ndcg_results = pd.DataFrame(ndcg_scores)

ndcg_results["ndcg_at_10"].describe()

count    610.000000
mean       0.075906
std        0.144208
min        0.000000
25%        0.000000
50%        0.000000
75%        0.099348
max        0.857981
Name: ndcg_at_10, dtype: float64

Like the other metrics, ndcg at 10 is fairly low. It appears as if personalization may be necessary for this recommendation system. 

# Rated-Only Precision@K Evaluation

Standard offline metrics treat every recommended movie that is not in the test positives as a miss. Rated-only precision is a complementary metric: it only evaluates recommendations that the user actually rated later. If none of a user's top-K recommendations were rated later, the score is missing rather than zero.

In [8]:
from src.metrics import rated_precision_at_k

k = 10
rated_precision_scores = []

test_users = test["userId"].unique()

for user_id in test_users:
    recs = recommend_popular_movies(
        train_ratings=train,
        movies=movies,
        user_id=user_id,
        k=k,
    )

    recommended_items = recs["movieId"].tolist()

    user_test = test[test["userId"] == user_id]
    rated_items = set(user_test["movieId"])
    positive_items = set(user_test[user_test["is_positive"]]["movieId"])

    rated_precision = rated_precision_at_k(
        recommended_items=recommended_items,
        positive_items=positive_items,
        rated_items=rated_items,
        k=k,
    )

    rated_precision_scores.append(
        {
            "userId": user_id,
            "rated_precision_at_10": rated_precision,
            "n_test_ratings": len(rated_items),
            "n_future_likes": len(positive_items),
        }
    )

rated_precision_results = pd.DataFrame(rated_precision_scores)

rated_precision_results["rated_precision_at_10"].describe()

count    221.000000
mean       0.771876
std        0.366386
min        0.000000
25%        0.500000
50%        1.000000
75%        1.000000
max        1.000000
Name: rated_precision_at_10, dtype: float64

Rated-only precision should be interpreted alongside the standard metrics. It avoids counting unobserved recommendations as misses, but it only evaluates users whose recommendations overlap with later ratings.

# Overall Baseline Metrics

In [9]:
baseline_summary = pd.DataFrame(
    {
        "metric": [
            "precision_at_10",
            "recall_at_10",
            "ndcg_at_10",
            "rated_precision_at_10",
        ],
        "mean": [
            precision_results["precision_at_10"].mean(),
            recall_results["recall_at_10"].mean(),
            ndcg_results["ndcg_at_10"].mean(),
            rated_precision_results["rated_precision_at_10"].mean(),
        ],
    }
)

baseline_summary

,metric,mean
0,precision_at_10,0.057869
1,recall_at_10,0.049672
2,ndcg_at_10,0.075906
3,rated_precision_at_10,0.771876


These are the numbers that we will hope to improve upon through this project